# Qwen3-VL-4B receipt extraction - run on Colab T4
**Before starting:** Runtime > Change runtime type > **T4 GPU**.
Put `VLMOCR_train.zip` (and later `VLMOCR_test.zip`) in your Google Drive folder `MyDrive/VLMOCR/`.
Outputs are written straight to Drive (`MyDrive/VLMOCR/outputs/<run>/`), so nothing is lost if Colab disconnects.
The script is resumable: after a disconnect run cells 2, 3 and the same run cell again - finished receipts are skipped.

In [ ]:
# 1. GPU check - must show a Tesla T4
!nvidia-smi

In [ ]:
# 2. Mount Drive and unpack the code + training images (no labels are uploaded)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/VLMOCR'
!mkdir -p /content/work
!unzip -q -o {DRIVE}/VLMOCR_train.zip -d /content/work
%cd /content/work
!ls

In [ ]:
# 3. Libraries (Qwen3-VL needs a recent transformers)
!pip install -q -U "transformers>=4.57" accelerate pillow

In [ ]:
# 4. SMOKE TEST: 5 training images. Check speed, valid=True and that the JSON looks right.
!python src/inference.py --images train/receipt_images --out {DRIVE}/outputs/smoke_test_gpu --model Qwen/Qwen3-VL-4B-Instruct --limit 5
import pandas as pd
print(pd.read_csv(f'{DRIVE}/outputs/smoke_test_gpu/timing.csv'))
# If the output is garbage / empty (fp16 overflow), re-run with:  --dtype bfloat16

In [ ]:
# 5. BASELINE on all 600 training images (resumable)
!python src/inference.py --images train/receipt_images --out {DRIVE}/outputs/train_baseline --model Qwen/Qwen3-VL-4B-Instruct

---
## FINAL TEST - do NOT run until the configuration is frozen
Only after the baseline/improvement work is finished and we agree the final settings.
Upload `VLMOCR_test.zip` to Drive first (it contains images only, no labels).

In [ ]:
# 6. FINAL TEST RUN (370 test images) - run only when told to
# !unzip -q -o {DRIVE}/VLMOCR_test.zip -d /content/work
# !python src/inference.py --images test/receipt_images --out {DRIVE}/outputs/test_final --model Qwen/Qwen3-VL-4B-Instruct